# Example Script
This script can be used as an easy way to implement the trained unet model.
To do it, you just need to download this script and run it

In [1]:
import torch
from imutils import paths
import numpy as np

In [2]:
# Remove the sample_data folder from google colab
!rm -rf /content/*
!git clone https://github.com/GiaHuyPham/DeepLearning_ComputerVision.git

Cloning into 'DeepLearning_ComputerVision'...
remote: Enumerating objects: 3046, done.
remote: Counting objects: 100% (791/791), done.
remote: Compressing objects: 100% (775/775), done.
remote: Total 3046 (delta 48), reused 23 (delta 14), pack-reused 2255 (from 3)
Receiving objects: 100% (3046/3046), 133.20 MiB | 41.51 MiB/s, done.
Resolving deltas: 100% (174/174), done.


# Setup File Directory

In [7]:
import shutil
import os

os.chdir("/content/DeepLearning_ComputerVision/CircleSegmentationUS_SpeckleImage")
os.getcwd()

'/content/DeepLearning_ComputerVision/CircleSegmentationUS_SpeckleImage'

# Device Agnostic

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load and Create an Instance for the UNET Model

In [11]:
MODEL_PATH = "output/model"
unet_model = torch.load(os.path.join(MODEL_PATH,"unet_model.pth"),weights_only=False)
unet_model.to(device)


Unet(
  (encoder): Encoder(
    (encBlocks): ModuleList(
      (0): Block(
        (block): Sequential(
          (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1))
        )
      )
      (1): Block(
        (block): Sequential(
          (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
        )
      )
      (2): Block(
        (block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
        )
      )
    )
    (maxPool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (decoder): Decoder(
    (upconv): ModuleList(
      (0): ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
      (1): ConvTranspose2d(32, 16, kernel_size=(2, 2), stride=(2, 2))
    )

# Implement Model on Unseen Data

In [12]:
# Predict Masks
from PIL import Image
import cv2
from torchvision import transforms
from torchvision.transforms import InterpolationMode


# Define image input shape
INPUT_IMAGE_HEIGHT = 128
INPUT_IMAGE_WIDTH = 128

# Write a transform for image
image_transform = transforms.Compose([
    # Transfrom to PILimage
    transforms.ToPILImage(),
    # Resize our image
    transforms.Resize(size=(INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH)),
    # Turn image into tensor
    transforms.ToTensor()
])

mask_transform = transforms.Compose([
    # Transfrom to PILimage
    transforms.ToPILImage(),
    # Resize our image
    transforms.Resize(size=(INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH),
                      interpolation=InterpolationMode.NEAREST),
    # Turn image into tensor
    transforms.ToTensor()
])

IMAGE_UNSEEN_DATASET = os.path.join("data_unseen","images_unseen")
imagePath_unseen = sorted(list(paths.list_images(IMAGE_UNSEEN_DATASET)))
pred_masks_path = "output/pred_masks"
os.makedirs(pred_masks_path)

for imgPath in imagePath_unseen:
  image_name = os.path.basename(imgPath)

  img = cv2.imread(imgPath,0)
  img = image_transform(img)
  img = img.unsqueeze(dim=1)
  img = img.to(device)

  unet_model.eval()
  with torch.inference_mode():
    mask_logits = unet_model(img).squeeze()
    pred_mask = torch.sigmoid(mask_logits)
    pred_mask = pred_mask.cpu().numpy()
    pred_mask = (pred_mask>0.5)*255
    pred_mask = pred_mask.astype(np.uint8)

  save_name = os.path.splitext(image_name)[0].replace('img_unseen', 'mask_unseen') + '.png'
  Image.fromarray(pred_mask).save(os.path.join(pred_masks_path, save_name))

  # print(f"Saved: {save_name}")

print("Done! All predicted masks saved.")

Done! All predicted masks saved.


## Plot

In [13]:
from pyunet.helperfunctions import make_frame_viewer
predMask_path = sorted(list(paths.list_images(pred_masks_path)))
make_frame_viewer(imagePath_unseen,predMask_path,image_transform,mask_transform)

interactive(children=(IntSlider(value=0, description='Frame:', max=379), Output()), _dom_classes=('widget-inte…

<function pyunet.helperfunctions.make_frame_viewer.<locals>.show_frame(iiframe)>